# Your grades, from the student portal

The CMS held course files. The student portal (SIS) holds your grades: the
transcript, and the detailed marks for every quiz and assignment.

We wrap two portal calls as tools and let an agent answer questions about your
grades.

One warning first. This portal is old and slow. It answers about once a minute,
and if you rush it, it returns an error. So we keep the calls few, and the
package waits and retries on its own. Be patient with it in class.

## 1. Log in

This works for **GIU** as well: put `GUC_SITE=giu` in your `.env` and use your
GIU username and password. Nothing else in the notebook changes.

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env

# One GUC login works for both the portal and the CMS. getpass hides the
# password so it is never saved in the notebook.
if not os.environ.get("GUC_USERNAME"):
    os.environ["GUC_USERNAME"] = input("GUC username: ")
if not os.environ.get("GUC_PASSWORD"):
    os.environ["GUC_PASSWORD"] = getpass.getpass("GUC password: ")

from guc_portal import GucPortal

portal = GucPortal()
print("asking the portal for your terms (this can take a few seconds)...")
seasons = portal.available_seasons()
print(len(seasons), "terms on record. Newest few:")
for value, label in seasons[:5]:
    print("  ", label)

## 2. Two tools

Two tools cover most questions:

- `list_my_semesters` : which terms do I have grades for
- `get_my_grades` : the detailed marks for one course in one term

We keep a tiny cache, so if the agent asks the same thing twice we do not hit the
slow portal again.

In [ ]:
from langchain.tools import tool

_grades_cache = {}


@tool
def list_my_semesters() -> list[str]:
    """List the terms the student has grades for, e.g. 'Winter 2024'."""
    return [label for _value, label in portal.available_seasons()]


@tool
def get_my_grades(semester: str, course: str) -> dict:
    """Get the detailed marks for one course in one term.

    `semester` is a term name like 'Winter 2024'. `course` is a course name or
    code like 'Discrete Math' or 'MATH501'. Returns each quiz/assignment mark
    (as earned / max) and the percentage of every course that term.
    """
    key = (semester.lower(), course.lower())
    if key not in _grades_cache:
        g = portal.get_grades_by_name(semester, course)
        _grades_cache[key] = {
            "course": g.course,
            "term": g.season,
            "items": [{"what": i.assessment, "grade": i.grade} for i in g.items],
            "course_percentages": g.percentages,
        }
    return _grades_cache[key]

## 3. Ask the agent

In [ ]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

agent = create_agent(
    model=model,
    tools=[list_my_semesters, get_my_grades],
    system_prompt=(
        "You help a GUC student read their grades. Use the tools. "
        "The portal is slow, so call each tool as few times as you can."
    ),
)


def ask(question):
    return agent.invoke({"messages": [HumanMessage(question)]})["messages"][-1].content


print(ask("How did I do in Discrete Math in Winter 2024? Which quiz was my weakest?"))

## What just happened, and your turn

The agent called `get_my_grades` once, read the per-quiz marks, and found the
weakest one. One slow call, not many.

Your turn:

- Ask about a different course or term.
- Add a `my_gpa` tool built on `portal.get_transcript_year(...)`.